# 05 — Deploy Endpoint + Monitoring (Model Monitor + CloudWatch)

This notebook demonstrates the “operability” parts of the system:

- Deploy latest **Approved** model from **Model Registry**
- Invoke the endpoint (show predictions)
- Enable **Data Capture**
- Create a **Model Monitor baseline** + monitoring schedule
- Create a basic **CloudWatch dashboard** (infrastructure monitoring)

This satisfies demo requirements:
- model registry
- endpoint invocation output
- monitoring reports
- infrastructure dashboards


In [10]:
%pip install -q -r ../docker/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [11]:
import json
import time
import boto3
import sagemaker
import pandas as pd

from sagemaker.model import ModelPackage
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)

In [12]:
# Load from previous notebook(s)
%store -r bucket
%store -r region
%store -r RUN_ID
%store -r MODEL_PACKAGE_GROUP
%store -r MODEL_PACKAGE_ARN

print("Bucket:", bucket)
print("Region:", region)
print("RUN_ID:", RUN_ID)
print("MODEL_PACKAGE_GROUP:", MODEL_PACKAGE_GROUP)
print("MODEL_PACKAGE_ARN:", MODEL_PACKAGE_ARN)

Bucket: sagemaker-us-east-1-318401170150
Region: us-east-1
RUN_ID: 20260223-010544
MODEL_PACKAGE_GROUP: buoycast-wave-models
MODEL_PACKAGE_ARN: arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/2


In [13]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
sm = boto3.client("sagemaker", region_name=region)

print("Role:", role)

Role: arn:aws:iam::318401170150:role/LabRole


In [14]:
import boto3

sm = boto3.client("sagemaker")

resp = sm.list_model_packages(
    ModelPackageGroupName="buoycast-wave-models",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)

for m in resp["ModelPackageSummaryList"]:
    print(m["ModelPackageArn"], "|", m["ModelApprovalStatus"])


arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/2 | PendingManualApproval
arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/1 | Approved


In [15]:
sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN,
    ModelApprovalStatus="Approved",
)


{'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/2',
 'ResponseMetadata': {'RequestId': '003c7402-1be3-41ef-8913-8c11abd20a46',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '003c7402-1be3-41ef-8913-8c11abd20a46',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '99',
   'date': 'Mon, 23 Feb 2026 01:36:11 GMT'},
  'RetryAttempts': 0}}

In [16]:
# Get latest APPROVED model package from the Model Package Group
resp = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if not resp["ModelPackageSummaryList"]:
    raise RuntimeError(
        "No APPROVED model packages found. "
        "Run Notebook 04, or approve a model in the Model Registry."
    )

model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print("ModelPackageArn:", model_package_arn)

ModelPackageArn: arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/2


In [17]:
# Deploy endpoint with data capture enabled
endpoint_name = f"buoycast-endpoint-{RUN_ID}"

data_capture_s3 = f"s3://{bucket}/buoycast/datacapture/{RUN_ID}/"
data_capture = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=data_capture_s3,
)

model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sess,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
    data_capture_config=data_capture,
)

print("Deployed endpoint:", endpoint_name)
print("Data capture S3:", data_capture_s3)

------!Deployed endpoint: buoycast-endpoint-20260223-010544
Data capture S3: s3://sagemaker-us-east-1-318401170150/buoycast/datacapture/20260223-010544/


In [18]:
import sagemaker
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

sess = sagemaker.Session()

predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)

print("Predictor bound to endpoint:", predictor.endpoint_name)


Predictor bound to endpoint: buoycast-endpoint-20260223-010544


In [19]:
import pandas as pd

sample_s3_uri = f"s3://{bucket}/buoycast/artifacts/{RUN_ID}/data/baseline/inference_sample.csv"
sample_df = pd.read_csv(sample_s3_uri)

payload = sample_df.to_csv(index=False, header=False)
pred = predictor.predict(payload)

print("Sample rows:", len(sample_df))
print(pred)


Sample rows: 10
{'predictions': [7.169437750473349, 3.404538789231407, 3.555424956030261, 7.232846322212388, 3.953076722510593, 7.601492033803968, 4.039382139764763, 9.082796653363665, 3.6480840982520513, 9.348251811375931]}


In [ ]:
# Model Monitor baseline + schedule (Data Quality monitor)

monitor_output_s3 = f"s3://{bucket}/buoycast/model-monitor/{RUN_ID}/"

baseline_s3_uri = f"s3://{bucket}/buoycast/artifacts/{RUN_ID}/data/baseline/baseline.csv"

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sess,
)

# 1) Suggest baseline (generates statistics + constraints)
baseline_job = monitor.suggest_baseline(
    baseline_dataset=baseline_s3_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"{monitor_output_s3}baseline/",
    wait=True,
)

print("Baseline statistics:", monitor.baseline_statistics())


import boto3

sm = boto3.client("sagemaker")

# Depending on SDK, suggest_baseline may return the job name or an object; handle both:
baseline_job_name = getattr(baseline_job, "baseline_job_name", None) or getattr(baseline_job, "job_name", None) or baseline_job
print("Baselining ProcessingJobName:", baseline_job_name)

job = sm.describe_processing_job(ProcessingJobName=baseline_job_name)

print("\nBaseline outputs (S3):")
for out in job["ProcessingOutputConfig"]["Outputs"]:
    s3uri = out["S3Output"]["S3Uri"]
    print(" -", out["OutputName"], "=>", s3uri)

# Optional: show exact constraints/statistics file locations
print("\nLook for these files under the output S3 URIs:")
print(" - constraints.json")
print(" - statistics.json")


INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-23-01-39-46-196


.........

In [ ]:
from sagemaker.model_monitor import Statistics, Constraints

statistics_s3_uri = s3uri + "statistics.json"
constraints_s3_uri = s3uri + "constraints.json"

statistics = Statistics.from_s3_uri(statistics_s3_uri)
constraints = Constraints.from_s3_uri(constraints_s3_uri)


In [ ]:
from sagemaker.model_monitor import EndpointInput, Constraints, Statistics
from sagemaker.model_monitor import CronExpressionGenerator

baseline_prefix = "s3://sagemaker-us-east-1-115800714036/buoycast/model-monitor/20260218-044750/baseline/"

from sagemaker.model_monitor import EndpointInput, CronExpressionGenerator

schedule_name = f"buoycast-data-quality-{RUN_ID}"

monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=endpoint_name,
        destination="/opt/ml/processing/input",
    ),
    output_s3_uri=f"{monitor_output_s3}reports/",
    statistics=statistics,
    constraints=constraints,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)



print("Created monitoring schedule:", schedule_name)
print("Monitor outputs:", monitor_output_s3)
print("Using statistics:", statistics_s3_uri)
print("Using constraints:", constraints_s3_uri)


In [ ]:
# CloudWatch dashboard (endpoint metrics)
cw = boto3.client("cloudwatch", region_name=region)

dashboard_name = f"BuoyCast-{RUN_ID}"

dashboard = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", endpoint_name, "VariantName", "AllTraffic"],
                    [".", "Invocation4XXErrors", ".", ".", ".", "."],
                    [".", "Invocation5XXErrors", ".", ".", ".", "."],
                ],
                "region": region,
                "stat": "Sum",
                "period": 300,
                "title": "Endpoint Invocations + Errors"
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "ModelLatency", "EndpointName", endpoint_name, "VariantName", "AllTraffic"],
                    [".", "OverheadLatency", ".", ".", ".", "."],
                ],
                "region": region,
                "stat": "Average",
                "period": 300,
                "title": "Endpoint Latency (ms)"
            }
        },
    ]
}

cw.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard),
)

print("Created/updated dashboard:", dashboard_name)

In [ ]:
# Save for cleanup notebook
%store endpoint_name
%store schedule_name
%store dashboard_name
%store data_capture_s3
%store monitor_output_s3